In [ ]:
# from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
# drive.mount('/content/gdrive', force_remount=True)


#base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/Final Project"
# base_dir = "/content/gdrive/MyDrive/Final Project"
base_dir = ""
# sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


In [ ]:
import importlib
import Models.GPT_Model as GPT_Model
import Datasets.DataLoader as DataLoader_Lib

importlib.reload(GPT_Model)
importlib.reload(DataLoader_Lib)

from Models.GPT_Model import GPT2_Lag, GPTConfig
from Datasets.DataLoader import TinyShakespeareDataLoader, TinyStoriesDataLoader, CombinedBinDataLoader

In [ ]:
# importlib.reload(GPT_Model)
# B = 4 
# block_size = 128

# loader = TinyStoriesDataLoader(max_length= block_size, batch_size=B)
# train_loader, val_loader = loader.get_data()


In [ ]:
importlib.reload(GPT_Model)
batch_size = 16 # Adjust batch size based on your Colab GPU memory
block_size = 128
warmup_steps = 20
num_steps_train = 3000
num_steps_val = 10
config = GPTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  lag_behind = 1,
  dropout = .1)

model = GPT2_Lag(config, device)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_steps_train
)

# Config = GPTConfig(num_heads = 12,
#   num_layers = 12,
#   vocab_size = 50257,
#   embedding_dim = 768,
#   block_size = block_size,
#   lag_behind = 1,
#   dropout = .1,
#   pad_token_id=loader.pad_token())

# model = GPT2_Lag(Config, device)
# model.to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [ ]:
train_loader = CombinedBinDataLoader('Datasets/combined_dataset.bin', batch_size, block_size, config, split='train')
val_loader = CombinedBinDataLoader('Datasets/combined_dataset.bin', batch_size, block_size, config, split='val')

In [ ]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10):
    model.eval()
    fwd_losses = torch.zeros(eval_iters)
    bwd_losses = torch.zeros(eval_iters)

    loader.cursor = 0
    
    for k in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)

        logits_pre, logits_fut, _ = model(x, y)

        V = model.config.vocab_size
        skip_dist = model.config.lag_behind

        fwd_losses[k] = F.cross_entropy(
            logits_pre.view(-1, V), y.view(-1)
        ).item()

        bwd_targets = x[:, :-skip_dist]
        bwd_logits  = logits_fut[:, skip_dist:]
        bwd_losses[k] = F.cross_entropy(
            bwd_logits.reshape(-1, V), bwd_targets.reshape(-1)
        ).item()

    model.train()
    avg_fwd = fwd_losses.mean().item()
    avg_bwd = bwd_losses.mean().item()
    ppl = torch.exp(torch.tensor(avg_fwd)).item()
    return avg_fwd, ppl, avg_bwd

# def get_lr(step, num_steps_train, warmup_steps=200, max_lr=3e-4, min_lr=3e-5):
#     if step < warmup_steps:
#         return max_lr * (step / warmup_steps)
#     return min_lr + 0.5 * (max_lr - min_lr) * (
#         1 + math.cos(math.pi * (step - warmup_steps) / (num_steps_train - warmup_steps))
#     )

tokens_per_step = train_loader.B * train_loader.SL
def train_loop(model, optimizer, scheduler, device, train_loader, val_loader,
               num_steps_train, num_steps_val, lam=0.5):

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
    print(f"Step    0 | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")
    start = time.time()
    tokens_seen = 0
    for step in range(num_steps_train):

        if step % 100 == 0 and step > 0:
            fwd_loss, ppl, bwd_loss = estimate_loss(model, val_loader, device, num_steps_val)
            print(f"Step {step:4d} | Val FWD: {fwd_loss:.4f} | PPL: {ppl:.2f} | Val BWD: {bwd_loss:.4f}")

        # lr = get_lr(step, num_steps_train)
        # for param_group in optimizer.param_groups:
        #     param_group['lr'] = lr

        x, y, _ = train_loader.get_data()
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        _, _, loss = model(x, y, lam=lam)
        loss.backward()
        optimizer.step()
        scheduler.step()
        tokens_seen += tokens_per_step
        if step % 10 == 0:
            stop = time.time()
            print(f"step {step:4d} | tokens {tokens_seen:,} | train loss {loss.item():.4f} | Time Since Last Train Print {(stop-start):.4f} seconds")
            start = time.time()

In [ ]:
train_loop(model, optimizer, scheduler, device, train_loader, val_loader,num_steps_train, num_steps_val)

In [ ]:
# batch = next(iter(train_loader))
# print(batch.keys())

In [ ]:
# @torch.inference_mode()
# def estimate_loss_and_perplexity(model, loader, device, eval_iters=10):
#     model.eval()

#     total_loss = 0.0
#     val_iter = iter(loader)

#     for _ in range(eval_iters):
#         try:
#             batch = next(val_iter)
#         except StopIteration:
#             val_iter = iter(loader)
#             batch = next(val_iter)

#         input_ids = batch["input_ids"].to(device)
#         labels = input_ids

#         _, _, loss = model(input_ids, labels)
#         total_loss += loss

#     avg_loss = total_loss.item() / eval_iters
#     perplexity = torch.exp(torch.tensor(avg_loss)).item()

#     model.train()
#     return avg_loss, perplexity

# def train_loop(model, optimizer, device, train_loader, val_loader, num_steps_train, num_steps_val):
#     print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

#     train_iter = iter(train_loader)  # create iterator once

#     for step in range(num_steps_train):
#         if step % 100 == 0:
#             val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, device, num_steps_val)
#             print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

#         # Restart iterator if all data has been seen
#         try:
#             batch = next(train_iter)
#         except StopIteration:
#             train_iter = iter(train_loader)
#             batch = next(train_iter)

#         optimizer.zero_grad()
#         input_ids = batch["input_ids"].to(device)
#         labels    = input_ids

#         _, _, loss = model(input_ids, labels)
#         loss.backward()
#         optimizer.step()

#         if step % 20 == 0:
#             print(f"Step {step:4d} train loss {loss.item():.4f}")
#     val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, device, num_steps_val)
#     print(f"Step {step:4d} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.4f}")

In [ ]:
# num_steps_train = 1000
# num_steps_val = 10
# train_loop(model, optimizer, device, train_loader, val_loader,num_steps_train, num_steps_val)